In [ ]:
# In this code, we did the mcQTSA Generation and use deepseek API to filter and format mcQTSA output.
# node7.jsonl is the final subsection file. And gpt_output/mcQTSA_new4.jsonl is the final mcQTSA file.

In [4]:
# mcQTSA Generation

import requests
import json, os
from pydantic import BaseModel
from typing import Optional, List
import time
from pydantic import BaseModel
from typing import Optional
from openai import OpenAI

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

OUTPUT_DIR = "gpt_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def call_llm(system_prompt: str,
             user_prompt: str,
             max_new_tokens: int = 1000,
             temperature: float = 0.1,
             model: str = "gpt-4.1-mini") -> str:
    try:
        #print(f"🌐 activate {model} by API (temperature={temperature})...")

        kwargs = dict(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            stream=False
        )

        if "gpt-5" in model or model.startswith("o1"):   
            kwargs["max_completion_tokens"] = max_new_tokens
        else:  
            kwargs["max_tokens"] = max_new_tokens

        response = client.chat.completions.create(**kwargs)

        #print(f"✅ finish activating model with API, token: {response.usage.total_tokens}")

        return response.choices[0].message.content

    except Exception as e:
        print(f"❌ Failed to use API: {e}")
        return None

# Set up prompt for multi-agents
# 1. Question Generator Agent (Ψ_Q)
system_prompt_agent_q = """YYou are an experienced professor of radio frequency integrated circuit design. Your task is to generate a single-choice question (MCQ) with exactly four options based on the provided textbook content.

CRITICAL REQUIREMENTS:
- The question MUST be strictly based on and answerable from the provided text content ONLY.
- The question should test understanding, not just recall. It can be about conceptual understanding, formula derivation, circuit analysis, or design methodology.
- Provide exactly four options (A, B, C, D). One option MUST be correct, and the other three MUST be plausible distractors derived from or related to the text.
- Distractors should be technically relevant but clearly incorrect upon close reading of the content.
- The question and options must be clear, unambiguous, and self-contained (does not require external knowledge).
- Avoid any reference to figures (e.g., “as shown in Figure X.Y”).
- Output format must be:

Question: <question text>
A. <option A>
B. <option B>
C. <option C>
D. <option D>
"""

# 2. Answer Generator Agent (Ψ_A)
system_prompt_agent_a = """You are a meticulous and patient radio frequency integrated circuit design tutor. You MUST follow these steps STRICTLY:

1.  **THINKING PROCESS** (<think> tags): 
    - First, conduct a thorough, step-by-step reasoning process inside <think> and </think> tags.
    - This is your internal monologue. Analyze the problem in detail: break down the question, identify key concepts, recall and summarize relevant principles, formulas, or methodologies from the provided reference text.
    - Plan your solution approach explicitly. Explain *why* you are taking each step, don't just list steps. This section should be comprehensive and exploratory.
2.  **STRUCTURED SOLUTION**: 
    - After your internal thinking, provide a formal, well-organized, and educational solution outside the think tags.
    - This section should translate your reasoning from the <think> section into a clear, logical narrative. Show all necessary calculations and derivations step by step, explaining the rationale behind each step based on the reference text.
    - Use headings, bullet points, or numbered steps for clarity. Ensure this part is detailed enough for a learner to follow the entire logic.
3.  **FINAL ANSWER** (<answer> tags): 
    - Conclude with a concise and precise final answer, wrapped in <answer> and </answer> tags.
    - This section should be brief, containing only the final numerical result, key conclusion, or summarized finding. It should not contain explanations or reasoning, which belong in the previous sections.

RULES:
- Your response MUST be based EXCLUSIVELY on the provided reference text. Do not use external knowledge.
- When applying concepts, you may reference equations from the text by describing their content (e.g., "using the equation for power gain that relates transconductance and load resistance"), but do not use equation numbers or figure references.
- The <think> and STRUCTURED SOLUTION sections must be substantially detailed and form the bulk of your response.
- The <answer> section must be succinct, containing only the final answer.
- Write in clear, academic English.
"""

# 3. Post-processor Agent (Ψ_P)
system_prompt_agent_p = """You are a quality assurance specialist for AI training data. 
Your primary responsibility is to minimally edit the raw QTSA outputs from Agents Q and A, 
ensuring valid formatting and JSON compliance, while preserving their content as faithfully as possible.

CRITICAL REQUIREMENTS:
1. **Minimal Editing**: 
   - Do NOT rewrite or paraphrase unless absolutely necessary for clarity or format.
   - Retain the original wording, technical details, equations, and reasoning from Agents Q and A.
   - If the raw outputs contain errors, only fix them in a minimal way (e.g., unmatched tags, broken equations).

2. **Output Format**: 
   - Output ONLY a valid JSON object. 
   - Do NOT wrap in code blocks (no ```json), no extra text, no explanations.

3. **Format Validation**: 
   - Ensure the JSON object contains all four QTSA components:
     - "question": the refined question
     - "thinking": the thinking process with <think>...</think>
     - "solution": the structured solution
     - "answer": the final answer with <answer>...</answer>
   - Check XML tags (<think>, <answer>) are properly opened and closed.

4. **Decontextualization**: 
   - Remove direct references to the original textbook source.
   - Example: Replace "according to Equation (Z)" with "according to the following equation".
   - Example: Replace "see Table A.B" with "see the following table".

5. **Refinement**: 
   - Only refine for grammar, clarity, and structure if absolutely necessary.
   - Do not shorten, simplify, or change the technical meaning of the content.

OUTPUT FORMAT (pure JSON only):
{
  "question": "...",
  "thinking": "<think> ... </think>",
  "solution": "...",
  "answer": "<answer> ... </answer>"
}

IMPORTANT: 
- Your output must be directly parseable by json.loads().
- Do NOT include markdown code blocks or any text outside the JSON.
"""


class Node(BaseModel):
    id: str  
    content: str  

class QTSAPair(BaseModel):
    question: str
    thinking: str
    solution: str
    answer: str
    node_id: str  
    sample_id: int
    raw_question: Optional[str] = None
    raw_answer: Optional[str] = None
    processed_output: Optional[str] = None

class AgentOutput(BaseModel):
    agent_q_output: str
    agent_a_output: str
    agent_p_output: str
    
# 3 agents

def generate_question(node: Node, sample_id: int = 0) -> str:
    """Agent Ψ_Q: Generate question from node content"""
    
    question_strategies = [
        "Ask a question that explores the fundamental concepts:",
        "Formulate a question requiring mathematical analysis or derivation:",
        "Pose a design-oriented question about implementation:",
        "Create an analytical question about performance aspects:",
        "Develop a question about practical applications or limitations:"
    ]

    strategy_index = (sample_id - 1) % len(question_strategies)
    strategy = question_strategies[strategy_index]
    
    user_prompt = f"""TEXTBOOK CONTENT:
{node.content}

{strategy}
If the content doesn't support this approach, adapt to a suitable perspective while maintaining technical depth."""
    
    question = call_llm(system_prompt_agent_q, user_prompt, temperature=1.0, max_new_tokens=500, model="gpt-4.1-mini")
    return question

def generate_answer(node: Node, question: str) -> str:
    """Agent Ψ_A: Generate answer with thinking process"""
    user_prompt = f"""REFERENCE TEXT:
{node.content}

QUESTION TO ANSWER:
{question}

Please provide a complete response following the required format."""
    
    answer = call_llm(system_prompt_agent_a, user_prompt, temperature=1, max_new_tokens=10000, model="gpt-5-mini")
    #if answer:
        #print(f"✅ Agent A 's answer: {answer[:100]}...")
    return answer

def post_process_data(raw_question: str, raw_answer: str) -> str:
    """Agent Ψ_P: Clean and format the QTSA data"""
    user_prompt = f"""RAW GENERATED DATA:
Question: {raw_question}
Answer: {raw_answer}

Please process this data according to the instructions."""
    
    processed_output = call_llm(system_prompt_agent_p, user_prompt, temperature=0.0, max_new_tokens=11000, model="gpt-4.1-mini")
    #if processed_output:
        #print(f"✅ Agent P 's answer: {processed_output[:100]}...")
    return processed_output


def process_single_node(node: Node, sample_id: int = 1) -> Optional[QTSAPair]:
    """process a single node, output entire QTSA and original output"""
    #print(f"Processing node {node.id}, sample #{sample_id + 1}")
    #print("=" * 60)

    # save the original output of each agents
    agent_outputs = AgentOutput(
        agent_q_output="",
        agent_a_output="", 
        agent_p_output=""
    )
    
    # Step 1: Generate question
    raw_question = generate_question(node, sample_id)
    if not raw_question:
        print(f"Failed to generate question for node {node.id}")
        return None
    agent_outputs.agent_q_output = raw_question
    
    # Step 2: generate answer
    raw_answer = generate_answer(node, raw_question)
    if not raw_answer:
        print(f"Failed to generate answer for node {node.id}")
        return None
    agent_outputs.agent_a_output = raw_answer
    
    # step 3: Post-process
    processed_output = post_process_data(raw_question, raw_answer)
    agent_outputs.agent_p_output = processed_output or ""
    
    try:
        if processed_output and processed_output.strip().startswith('{'):
            data_dict = json.loads(processed_output)
            qtsa_pair = QTSAPair(
                question=data_dict.get("question", raw_question),
                thinking=data_dict.get("thinking", ""),
                solution=data_dict.get("solution", ""),
                answer=data_dict.get("answer", ""),
                node_id=node.id,
                sample_id=sample_id,
                raw_question=raw_question,
                raw_answer=raw_answer,
                processed_output=processed_output
            )
        else:
           
            qtsa_pair = QTSAPair(
                question=raw_question,
                thinking=raw_answer,
                solution=raw_answer,
                answer=raw_answer,
                node_id=node.id,
                sample_id=sample_id,
                raw_question=raw_question,
                raw_answer=raw_answer,
                processed_output=processed_output or "No JSON output"
            )
        
        #print(f"✅ paragraph {node.id} is finished ")

        save_agent_outputs(agent_outputs, node.id, sample_id)
        
        return qtsa_pair
        
    except json.JSONDecodeError as e:
        print(f"❌ Fail to parse the file: {e}")
        return QTSAPair(
            question=raw_question,
            thinking=raw_answer,
            solution=raw_answer,
            answer=raw_answer,
            node_id=node.id,
            sample_id=sample_id,
            raw_question=raw_question,
            raw_answer=raw_answer,
            processed_output=f"JSON Parse Error: {e}"
        )

def save_agent_outputs(agent_outputs: AgentOutput, node_id: str, sample_id: int):
    agent_filename = f"{OUTPUT_DIR}/mcQTSA_agent.jsonl"                # output file
    
    output_data = {
        "node_id": node_id,
        "sample_id": sample_id,
        "agent_q_output": agent_outputs.agent_q_output,
        "agent_a_output": agent_outputs.agent_a_output,
        "agent_p_output": agent_outputs.agent_p_output
    }
    
    with open(agent_filename, 'a', encoding='utf-8') as f:
        f.write(json.dumps(output_data, ensure_ascii=False) + '\n')
    
    #print(f"💾 Original outputs are saved as: {agent_filename}")

def process_multiple_samples(node: Node, num_samples: int = 3) -> List[QTSAPair]:
    """Process multiple samples for a single node"""
    results = []
    
    print(f"Processing node {node.id} with {num_samples} samples...")
    
    for sample_id in range(1, num_samples + 1): 
        result = process_single_node(node, sample_id)
        if result:
            results.append(result)
            print(f"  Sample #{sample_id} completed")
        else:
            print(f"  Sample #{sample_id} failed")
        
        # Add delay between samples
        if sample_id < num_samples - 1:
            time.sleep(2)
    
    #print(f"Node {node.id} completed: {len(results)}/{num_samples} samples successful")
    return results

def save_results(results: List[QTSAPair], filename: str = None):
    """Save results to JSONL file"""
    if not results:
        print("No results to save")
        return
    
    with open(filename, 'a', encoding='utf-8') as f:
        for result in results:
            clean_data = {
                "node_id": result.node_id,
                "sample_id": result.sample_id,
                "question": result.question,
                "thinking": result.thinking,
                "solution": result.solution,
                "answer": result.answer
            }
            f.write(json.dumps(clean_data, ensure_ascii=False) + '\n')
    
    #print(f"Results saved to {filename}")

# Test multiple samples for one node
if __name__ == "__main__":
    nodes = []
    try:
        with open('node7.jsonl', 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                nodes.append(Node(id=data["id"], content=data["text"]))
        
        print(f"Loaded {len(nodes)} nodes")

        num_samples = 5   

        for idx, selected_node in enumerate(nodes):
            #print(f"\n=== Processing node #{idx + 1}: {selected_node.id} ===")

            results = process_multiple_samples(selected_node, num_samples)

            if results:
                filename = os.path.join(
                    OUTPUT_DIR, "mcQTSA.jsonl"          # output
                )
                save_results(results, filename)

                print(f"Generated {len(results)} QTSA pairs -> saved to {filename}")
            else:
                print("No results were generated for this node")

        print(f"All processing completed. Files saved to {OUTPUT_DIR}")

    except FileNotFoundError:
        print("Error: node7.jsonl file not found")
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

Loaded 1108 nodes
Processing node 1 with 5 samples...
  Sample #1 completed
  Sample #2 completed
  Sample #3 completed
  Sample #4 completed
  Sample #5 completed
Generated 5 QTSA pairs -> saved to gpt_output/mcQTSA.jsonl
Processing node 2 with 5 samples...
  Sample #1 completed
  Sample #2 completed
  Sample #3 completed
  Sample #4 completed
  Sample #5 completed
Generated 5 QTSA pairs -> saved to gpt_output/mcQTSA.jsonl
Processing node 3 with 5 samples...
  Sample #1 completed
  Sample #2 completed
  Sample #3 completed
  Sample #4 completed
  Sample #5 completed
Generated 5 QTSA pairs -> saved to gpt_output/mcQTSA.jsonl
Processing node 4 with 5 samples...
  Sample #1 completed
  Sample #2 completed
  Sample #3 completed
  Sample #4 completed
  Sample #5 completed
Generated 5 QTSA pairs -> saved to gpt_output/mcQTSA.jsonl
Processing node 5 with 5 samples...
  Sample #1 completed
  Sample #2 completed
  Sample #3 completed
  Sample #4 completed
  Sample #5 completed
Generated 5 QTSA

In [9]:
import json
import re

qtsa_file = "gpt_output/mcQTSA.jsonl"
agent_file = "gpt_output/mcQTSA_agent.jsonl"
output_file = "gpt_output/mcQTSA_new.jsonl"

agent_pairs = set()
with open(agent_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        agent_pairs.add((item["node_id"], item["sample_id"]))

new_data = []
problem_entries = []

think_pattern = re.compile(r"<think>.*?</think>", re.DOTALL)
answer_pattern = re.compile(r"<answer>.*?</answer>", re.DOTALL)

with open(qtsa_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        key = (item["node_id"], item["sample_id"])

        if key not in agent_pairs: 
            full_text = json.dumps(item, ensure_ascii=False)

            think_match = think_pattern.search(full_text)
            answer_match = answer_pattern.search(full_text)

            if think_match and answer_match:
                think_text = think_match.group(0)
                answer_text = answer_match.group(0)

                solution_text = full_text[
                    full_text.find(think_text) + len(think_text) : full_text.find(answer_text)
                ]

                solution_text = solution_text.replace("STRUCTURED SOLUTION", "").strip()

                item["thinking"] = think_text
                item["solution"] = solution_text
                item["answer"] = answer_text

                problem_entries.append(key)

        new_data.append(item)

print("fixed sample (node_id, sample_id):")
for key in problem_entries:
    print(key)

with open(output_file, "w", encoding="utf-8") as f:
    for item in new_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n reselts are saved as {output_file}")


修复的条目 (node_id, sample_id):
('18', 5)
('243', 1)
('261', 2)
('283', 5)
('477', 3)
('484', 3)
('510', 5)

修复完成，结果已保存到 gpt_output/mcQTSA_new.jsonl


In [6]:
import json
from openai import OpenAI
import time

# Initialize DeepSeek API
client = OpenAI(
    api_key="YOUR_API_KEY",
    base_url="https://api.deepseek.com",
)

input_file = "gpt_output/mcQTSA_new.jsonl"
output_file = "gpt_output/mcQTSA_new1.jsonl"

def clean_question_with_deepseek(question: str) -> str:
    prompt = f"""You are a data cleaning assistant. The following text is a multiple-choice question.

Your task:
- KEEP the entire question stem and ALL options, exactly as they are written.
- REMOVE ONLY the final line that explicitly reveals the correct answer 
  (e.g., "Answer: C", "Correct answer is D", "The right choice is ...").
- Do not remove or modify any lines that are options (like choices starting with A, B, C, D, etc.).
- If no final answer line is present, return exactly "0".
- Do not rewrite or paraphrase. Just return the cleaned text.

Question text:
{question}

Return only the cleaned question (if changed), or "0" if no change is needed."""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"DeepSeek API error: {e}")
        return "0"  # fallback

with open(input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:

    for idx, line in enumerate(fin, 1):
        item = json.loads(line)
        if "question" in item:
            original_q = item["question"]
            cleaned_q = clean_question_with_deepseek(original_q)

            # Update only if DeepSeek actually modified it
            if cleaned_q != "0" and cleaned_q != original_q:
                item["question"] = cleaned_q
                changed = True
            else:
                changed = False

        fout.write(json.dumps(item, ensure_ascii=False) + "\n")

        # Print first few samples for inspection
        if idx <= 5:
            if not changed:
                print(f"\nSample {idx}: No change (kept original)\n{'-'*60}")
            else:
                print(f"\nSample {idx}: Modified\nOriginal:\n{original_q}\nCleaned:\n{cleaned_q}\n{'-'*60}")

        time.sleep(0.5)  # avoid rate limits

print(f"✅ Processing complete. Output saved to {output_file}")


Sample 1: No change (kept original)
------------------------------------------------------------

Sample 2: No change (kept original)
------------------------------------------------------------

Sample 3: No change (kept original)
------------------------------------------------------------

Sample 4: No change (kept original)
------------------------------------------------------------

Sample 5: No change (kept original)
------------------------------------------------------------
✅ Processing complete. Output saved to gpt_output/mcQTSA_new1.jsonl


In [7]:
import json
from openai import OpenAI
import time

# Initialize DeepSeek API
client = OpenAI(
    api_key="YOUR_API_KEY",
    base_url="https://api.deepseek.com",
)

input_file = "gpt_output/mcQTSA_new1.jsonl"
output_file = "gpt_output/mcQTSA_new2.jsonl"

def extract_answer_with_deepseek(answer_text: str) -> str:
    prompt = f"""Extract and format multiple-choice answer options from: {answer_text}

Rules:
- Extract only option letters (A,B,C,D)
- Remove all descriptive text
- Format as: <answer>X</answer> or <answer>XY</answer>
- Sort multiple options alphabetically
- Return only the formatted string"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
                
    except Exception as e:
        print(f"DeepSeek API error: {e}")
        return "<answer></answer>"

def get_answer_text(item):
    """Extract answer text from JSON item"""
    if "answer" in item:
        if isinstance(item["answer"], str):
            return item["answer"]
        elif isinstance(item["answer"], dict):
            for key in ['answer', 'correct_answer', 'correct', 'text']:
                if key in item["answer"] and isinstance(item["answer"][key], str):
                    return item["answer"][key]
    return ""

with open(input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:

    for idx, line in enumerate(fin, 1):
        item = json.loads(line)
        original_answer = get_answer_text(item)
        
        if original_answer:
            formatted_answer = extract_answer_with_deepseek(original_answer)
            # Update the answer field
            if isinstance(item["answer"], str):
                item["answer"] = formatted_answer
            elif isinstance(item["answer"], dict):
                updated = False
                for key in ['answer', 'correct_answer', 'correct']:
                    if key in item["answer"]:
                        item["answer"][key] = formatted_answer
                        updated = True
                        break
                if not updated:
                    item["answer"]["answer"] = formatted_answer
                    
            changed = formatted_answer != original_answer
        else:
            changed = False
            formatted_answer = "No answer found"

        fout.write(json.dumps(item, ensure_ascii=False) + "\n")

        # Print first few samples for inspection
        if idx <= 10:
            status = "Formatted" if changed else "No change"
            print(f"Sample {idx}: {status}")
            if changed:
                print(f"Original: {original_answer}")
            print(f"Result: {formatted_answer}")
            print("-" * 40)

        time.sleep(0.5)  # Rate limiting

print(f"✅ Processing complete. Output saved to {output_file}")

Sample 1: No change
Result: <answer>B</answer>
----------------------------------------
Sample 2: Formatted
Original: <answer> B </answer>
Result: <answer>B</answer>
----------------------------------------
Sample 3: Formatted
Original: <answer>
B
</answer>
Result: <answer>B</answer>
----------------------------------------
Sample 4: No change
Result: <answer>B</answer>
----------------------------------------
Sample 5: No change
Result: <answer>B</answer>
----------------------------------------
Sample 6: No change
Result: <answer>B</answer>
----------------------------------------
Sample 7: No change
Result: <answer>B</answer>
----------------------------------------
Sample 8: Formatted
Original: <answer> C </answer>
Result: <answer>C</answer>
----------------------------------------
Sample 9: No change
Result: <answer>A</answer>
----------------------------------------
Sample 10: No change
Result: <answer>A</answer>
----------------------------------------
✅ Processing complete. Out

In [2]:
import json

def replace_answers(original_file, answer_file, output_file):
    """
    Replace the 'answer' field in original_file with the 'answer' field from answer_file
    Keep other fields unchanged and output to output_file
    """
    try:
        # Read both files
        with open(original_file, 'r', encoding='utf-8') as f1, \
             open(answer_file, 'r', encoding='utf-8') as f2:
            
            original_lines = f1.readlines()
            answer_lines = f2.readlines()
        
        # Check if line counts match
        if len(original_lines) != len(answer_lines):
            print(f"Warning: Line count mismatch - original: {len(original_lines)}, answer: {len(answer_lines)}")
        
        # Process data
        output_lines = []
        for i, (orig_line, ans_line) in enumerate(zip(original_lines, answer_lines)):
            try:
                # Parse JSON
                orig_data = json.loads(orig_line.strip())
                ans_data = json.loads(ans_line.strip())
                
                # Replace answer field
                if 'answer' in ans_data:
                    orig_data['answer'] = ans_data['answer']
                
                # Write new line
                output_lines.append(json.dumps(orig_data, ensure_ascii=False))
                
            except json.JSONDecodeError:
                continue
        
        # Write output file
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for line in output_lines:
                f_out.write(line + '\n')
        
        print(f"Processed {len(output_lines)} lines")
        print(f"Output saved to: {output_file}")
        
    except FileNotFoundError as e:
        print(f"File not found: {e}")
    except Exception as e:
        print(f"Error: {e}")

# Usage
if __name__ == "__main__":
    original_file = "gpt_output/mcQTSA_new.jsonl"
    answer_file = "gpt_output/mcQTSA_new2.jsonl"
    output_file = "gpt_output/mcQTSA_new3.jsonl"
    
    replace_answers(original_file, answer_file, output_file)

Processed 5540 lines
Output saved to: gpt_output/mcQTSA_new3.jsonl


In [1]:
# add accurate choice to answer (in QTSA generation p_agent did not act well)

import json
import os
import re
from openai import OpenAI
from tqdm import tqdm

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# --- File paths ---
agent_file = "gpt_output/mcQTSA_agent.jsonl"
new_file = "gpt_output/mcQTSA_new3.jsonl"
output_file = "gpt_output/mcQTSA_new3_fixed.jsonl"

# --- Step 1: Load agent data into a dictionary ---
agent_map = {}
with open(agent_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        item = json.loads(line)
        key = (item["node_id"], item["sample_id"])
        agent_map[key] = item["agent_q_output"]

# --- Step 2: Helper to detect whether question already has options ---
def has_options(question: str) -> bool:
    """
    Return True if the question already includes multiple-choice options.
    Detects patterns like 'A)', 'A.', 'A、', '(A)', etc.
    """
    return bool(re.search(r'\b[A-D][).、]', question)) or bool(re.search(r'\([A-D]\)', question))

# --- Step 3: Define GPT function for repairing incomplete questions ---
def fix_question(agent_q, question):
    prompt = f"""
You are a question format repair assistant.

Two fields are given:
---
agent_q_output:
{agent_q}
---
question:
{question}
---

Task:
1. The question field might be missing multiple-choice options.
2. If the question already contains options (A, B, C, D, etc.), keep it unchanged.
3. If it lacks options, rebuild the full question by combining the stem with options extracted from agent_q_output.
4. Remove any answer text like "Answer: B" or "Correct answer is D".
Return ONLY the repaired question text. Do not explain.
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        fixed_q = response.choices[0].message.content.strip()
        return fixed_q
    except Exception as e:
        print("❌ Error:", e)
        return question  # fallback

# --- Step 4: Process all records in new3 file ---
fixed_data = []
changed_indices = []

with open(new_file, "r", encoding="utf-8") as f:
    for index, line in enumerate(tqdm(f, desc="Processing questions"), start=1):
        if not line.strip():
            continue
        item = json.loads(line)
        key = (item["node_id"], item["sample_id"])
        q_text = item.get("question", "")

        # Step 4.1: If question already has options, skip
        if has_options(q_text):
            fixed_data.append(item)
            continue

        # Step 4.2: Otherwise, call GPT to fix
        agent_q = agent_map.get(key)
        if agent_q:
            new_q = fix_question(agent_q, q_text)
            item["question"] = new_q
            print(f"Question {index} repaired by GPT.")
            changed_indices.append(index)
        fixed_data.append(item)

# --- Step 5: Save the repaired file ---
with open(output_file, "w", encoding="utf-8") as f:
    for item in fixed_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n✅ Finished! Total {len(fixed_data)} items processed.")
print(f"Questions repaired by GPT: {len(changed_indices)}")


Processing questions: 16it [00:04,  3.74it/s]

Question 16 repaired by GPT.


Processing questions: 19it [00:06,  2.62it/s]

Question 19 repaired by GPT.


Processing questions: 33it [00:08,  4.17it/s]

...

Processing questions: 5520it [20:59,  1.70it/s]

Question 5520 repaired by GPT.


Processing questions: 5540it [21:01,  4.39it/s]

Question 5534 repaired by GPT.



✅ Finished! Total 5540 items processed.
Questions repaired by GPT: 634


In [2]:
# 📊 Summary of modified questions (only works if 'changed_indices' still exists in memory)

if "changed_indices" in locals():
    total_changed = len(changed_indices)
    print(f"✅ Total modified questions: {total_changed}\n")
    
    if total_changed > 0:
        print("Modified question indices (1-based):")
        print(", ".join(map(str, changed_indices)))
    else:
        print("No questions were modified by GPT.")
else:
    print("⚠️ Variable 'changed_indices' not found. Please rerun the repair cell first.")


✅ Total modified questions: 634

Modified question indices (1-based):
16, 19, 33, 38, 39, 47, 59, 64, 74, 86, 87, 88, 93, 98, 99, 101, 103, 110, 123, 161, 164, 167, 168, 172, 174, 175, 183, 197, 200, 205, 207, 211, 219, 221, 237, 242, 246, 247, 258, 271, 284, 318, 321, 323, 349, 350, 354, 361, 408, 416, 418, 419, 434, 435, 448, 454, 456, 461, 470, 499, 502, 506, 511, 522, 533, 551, 553, 598, 628, 634, 636, 681, 696, 698, 699, 700, 714, 720, 725, 734, 771, 808, 811, 813, 815, 820, 834, 848, 875, 876, 883, 904, 924, 943, 944, 945, 975, 980, 1001, 1013, 1024, 1048, 1054, 1058, 1075, 1079, 1092, 1109, 1120, 1121, 1124, 1158, 1164, 1212, 1225, 1255, 1259, 1260, 1261, 1274, 1299, 1300, 1314, 1318, 1332, 1346, 1353, 1355, 1368, 1369, 1379, 1380, 1381, 1385, 1403, 1410, 1423, 1424, 1427, 1431, 1434, 1441, 1443, 1455, 1460, 1462, 1463, 1464, 1479, 1482, 1483, 1484, 1486, 1501, 1503, 1506, 1508, 1509, 1511, 1512, 1514, 1515, 1516, 1527, 1531, 1533, 1541, 1554, 1555, 1557, 1559, 1563, 1569, 1577,

In [3]:
import json

def add_ids_to_jsonl(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        
        id_counter = 1
        
        for line in infile:
            line = line.strip()
            if line: 
                try:
                    data = json.loads(line)
                    data['id'] = id_counter
                    id_counter += 1

                    json.dump(data, outfile, ensure_ascii=False)
                    outfile.write('\n')
                    
                except json.JSONDecodeError as e:
                    print(f"解析JSON错误在行 {id_counter}: {e}")
    
    print(f"Finish processing！Totally solved {id_counter-1} samples")
    print(f"output file: {output_file}")

input_file = "gpt_output/mcQTSA_new3_fixed.jsonl"
output_file = "gpt_output/mcQTSA_new4.jsonl"

add_ids_to_jsonl(input_file, output_file)

处理完成！共处理了 5540 条记录
输出文件: gpt_output/mcQTSA_new4.jsonl
